In [2]:
!pip install opencv-python
!pip install numpy
import cv2
import numpy as np


[notice] A new release of pip is available: 23.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [3]:
img = cv2.imread('/Users/noelnegron/Desktop/Observational Astronomy /Automatic_Crater_Depth/Pupin.png')
gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
cv2.imshow("Grey", gray_img)
cv2.waitKey(0)
cv2.destroyAllWindows()

2025-05-19 17:20:35.646 python[7440:22568001] +[IMKClient subclass]: chose IMKClient_Modern
2025-05-19 17:20:35.646 python[7440:22568001] +[IMKInputSession subclass]: chose IMKInputSession_Modern


In [4]:
# Convert image to HSV
hsv_img = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

# Define HSV range for black (shadows)
lower_black = np.array([0, 0, 0])
upper_black = np.array([180, 255, 50])

# Create mask
mask = cv2.inRange(hsv_img, lower_black, upper_black)

# Find contours
mask_contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

if mask_contours:
    # Find the largest contour
    largest_contour = max(mask_contours, key=cv2.contourArea)

    # Get bounding rectangle
    x, y, w, h = cv2.boundingRect(largest_contour)

    # Draw the bounding box
    cv2.rectangle(img, (x, y), (x + w, y + h), (0, 0, 255), 2)

    # Prepare text
    text = f"W:{w}px H:{h}px"
    print(f"Width: {w} pixels")
    print(f"Height: {h} pixels")

    # Draw text on image (just above the rectangle)
    cv2.putText(img, text, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 
                0.6, (255, 255, 255), 2, cv2.LINE_AA)

# Show result
cv2.imshow("Black Mask", mask)
cv2.imshow("Shadow Detected", img)
cv2.waitKey(0)
cv2.destroyAllWindows()

Width: 135 pixels
Height: 74 pixels


In [5]:
#derive length of shadow using known propotions:
reference_kilometers = 93
km_per_pixel = reference_kilometers / w

print(f"Scale: 1 pixel = {km_per_pixel:.4f} kilometers")

# Calculate the length of the shadow in kilometers
width_km = w * km_per_pixel
height_km = h * km_per_pixel

label = f"W:{width_km:.1f}km H:{height_km:.1f}km"
print(label)

Scale: 1 pixel = 0.6889 kilometers
W:93.0km H:51.0km


In [6]:
#Calculate the sun angle
#Step 1: get the subsolar point using JBL Horizions 
#This is the point on the Moon's surface where the Sun's rays are perpendicular. 
#You can find the subsolar point's coordinates using JPL Horizons.
!pip3 install requests
import requests
import re


[notice] A new release of pip is available: 23.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [7]:
import requests
import re
from datetime import datetime, timedelta

def get_subsolar_coords(start_time):
    """
    Fetches the sub-solar longitude and latitude for the Moon at a given UTC time.

    Parameters:
        start_time (str): A UTC datetime string in the format 'YYYY-MM-DD HH:MM'

    Prints:
        Timestamp, SunSub-LON, and SunSub-LAT if successful.
    """
    # Parse the start time string
    try:
        start_dt = datetime.strptime(start_time, '%Y-%m-%d %H:%M')
    except ValueError:
        print("Error: start_time must be in 'YYYY-MM-DD HH:MM' format.")
        return

    # Add 1 minute to get the stop time
    stop_dt = start_dt + timedelta(minutes=1)
    stop_time = stop_dt.strftime('%Y-%m-%d %H:%M')

    # Define API base URL
    base_url = 'https://ssd.jpl.nasa.gov/api/horizons.api'

    # Define parameters
    params = {
        'format': 'json',
        'COMMAND': "'301'",
        'OBJ_DATA': 'NO',
        'MAKE_EPHEM': 'YES',
        'EPHEM_TYPE': 'OBSERVER',
        'CENTER': "'500@399'",
        'START_TIME': f"'{start_time}'",
        'STOP_TIME': f"'{stop_time}'",
        'STEP_SIZE': "'1 d'",
        'QUANTITIES': "'15'"  # Sub-solar longitude and latitude
    }

    # Make the request
    response = requests.get(base_url, params=params)
    print("Request URL:", response.url)

    if response.status_code == 200:
        data = response.json()
        result_text = data.get("result", "")

        # Extract SunSub-LON and SunSub-LAT
        match = re.search(
            r"\$\$SOE\s*\n\s*(\d{4}-\w{3}-\d{2} \d{2}:\d{2})\s+([\d\.\-]+)\s+([\d\.\-]+)",
            result_text
        )

        if match:
            timestamp = match.group(1)
            sunsub_lon = float(match.group(2))
            sunsub_lat = float(match.group(3))
            print(f"Timestamp     : {timestamp}")
            print(f"SunSub-LON    : {sunsub_lon}")
            print(f"SunSub-LAT    : {sunsub_lat}")
            return sunsub_lon, sunsub_lat
        else:
            print("Sun-subsolar coordinates not found in response.")
    else:
        print("HTTP Error:", response.status_code)

In [8]:
#TRY TO CALCULATE SUB-SOLAR POINTS OF THE MOON
import math
sunsub_lon, sunsub_lat = get_subsolar_coords("2025-03-09 00:05")
crater_lat , crater_long = 9.62, -20.08

def solar_zenith_angle(lat1_deg, lon1_deg, lat2_deg, lon2_deg):
    """
    Calculate the angular distance (solar zenith angle) between two points 
    on a sphere using the spherical law of cosines.

    Parameters:
    - lat1_deg, lon1_deg: Latitude and longitude of the observation point (e.g. crater)
    - lat2_deg, lon2_deg: Latitude and longitude of the sub-solar point

    Returns:
    - zenith_angle_deg: Solar zenith angle in degrees
    """
    # Convert degrees to radians
    lat1 = math.radians(lat1_deg)
    lon1 = math.radians(lon1_deg)
    lat2 = math.radians(lat2_deg)
    lon2 = math.radians(lon2_deg)

    # Spherical law of cosines
    cos_theta = (math.sin(lat1) * math.sin(lat2) +
                 math.cos(lat1) * math.cos(lat2) * math.cos(lon1 - lon2))

    # Clamp result to valid range for acos to avoid rounding errors
    cos_theta = min(1.0, max(-1.0, cos_theta))

    # Compute angle in radians then convert to degrees
    theta_rad = math.acos(cos_theta)
    theta_deg = math.degrees(theta_rad)

    return theta_deg

Request URL: https://ssd.jpl.nasa.gov/api/horizons.api?format=json&COMMAND=%27301%27&OBJ_DATA=NO&MAKE_EPHEM=YES&EPHEM_TYPE=OBSERVER&CENTER=%27500%40399%27&START_TIME=%272025-03-09+00%3A05%27&STOP_TIME=%272025-03-09+00%3A06%27&STEP_SIZE=%271+d%27&QUANTITIES=%2715%27
Timestamp     : 2025-Mar-09 00:05
SunSub-LON    : 67.073299
SunSub-LAT    : -0.246615


In [9]:
#sun_angle calcualtions:
solar_zenith_angle = solar_zenith_angle(crater_lat, crater_long, sunsub_lat, sunsub_lon)
#This is the angle between the sun and the crater
sunangle = 90 - solar_zenith_angle
print(sunangle)

2.7653505267907263


In [10]:
import math 
# Calculate the depth of the crater using the formula
# depth = (shadow_length * tan(sun_angle))
shadow_length = height_km
depth = shadow_length * math.tan(math.radians(sunangle))
print(f"Depth of the crater: {depth:.2f} km")


Depth of the crater: 2.46 km
